# Hallucination & Alignment — from scratch

Why fluent language models produce confident falsehoods, and what the defenses actually do — built
and **proven** on tiny, deterministic toy distributions. Every function here is imported from the
companion script `hallucination_alignment.py`, so the numbers in this notebook, on the concept page,
and in the figures are computed in exactly one place and **cannot drift**.

Read the `assert` statements as the contract: each demo states its qualitative claim *before* it
prints the number, so a passing run is a proof, not a vibe.

**The four demos**
1. **temperature → hallucination** — a toy "knowledge" sampler; hotter decoding emits more wrong tokens.
2. **grounding → fewer hallucinations** — a retrieval boost concentrates mass on the supported answer.
3. **confidence → calibrated abstention** — answer only when confident; trade coverage for accuracy.
4. **helpful vs harmless** — one refusal knob can't maximize both: the alignment Pareto frontier.

> The one-sentence core: an LLM samples the next token from a softmax that **never** assigns zero
> probability to a wrong answer — so hallucination is the *base rate* of an ungrounded sampler, and
> every fix either **moves mass onto truth** (grounding, tuning) or **refuses the low-confidence tail**
> (calibration, abstention).

## Setup — device honesty and the single source of truth

We import the constants and functions from `hallucination_alignment.py`. The logic is
device-independent; we pin the reproducible trace to **CPU** and print the device **honestly**, so the
printed device always matches where the numbers were actually produced.

In [1]:
import numpy as np
import torch
import torch.nn.functional as F

# Everything is imported from the ONE seeded source of truth — no logic is redefined here.
from hallucination_alignment import (
    ANSWER_VOCAB, SUPPORTED_IDX, BASE_LOGITS, GROUNDING_BOOST, TEMPS_HALLUCINATION,
    PERMISSIVENESS_THRESHOLDS,
    softmax_with_temperature, base_logits_tensor, grounded_logits_tensor,
    hallucination_rate_curve,
    make_calibration_data, coverage_accuracy_curve, expected_calibration_error,
    make_request_population, helpful_harmless_curve,
)

DEVICE = "cpu"  # pin the trace to CPU for reproducibility
detected = ("cuda" if torch.cuda.is_available()
            else "mps" if torch.backends.mps.is_available() else "cpu")
print(f"device: {DEVICE} (detected {detected}; pinned to CPU for reproducibility)")
print("torch:", torch.__version__, " | numpy:", np.__version__)
print("answer vocab:", ANSWER_VOCAB, " | supported =", ANSWER_VOCAB[SUPPORTED_IDX])

device: cpu (detected mps; pinned to CPU for reproducibility)
torch: 2.12.0  | numpy: 2.4.6
answer vocab: ('Austen', 'Bronte', 'Dickens', 'Eliot', 'Hardy', 'Gaskell')  | supported = Austen


## The softmax floor: there is no "assign zero" option

A language model turns logits into a distribution with the softmax $p_i = e^{z_i}/\sum_j e^{z_j}$.
Because every $e^{z_j} > 0$, **every** token keeps non-zero probability — including every wrong answer.
That residual mass on wrong tokens is the structural seed of hallucination. Our toy "knowledge"
distribution: the model knows the answer (`Austen` has the largest logit) but the distractors still
carry real, samplable mass.

In [2]:
logits = base_logits_tensor(device=DEVICE)
probs = softmax_with_temperature(logits, 1.0)

# The supported token wins, but EVERY wrong token has > 0 probability — assert it, then show it.
assert (probs > 0).all(), "softmax assigns non-zero mass to every token — no 'impossible' option"
assert int(torch.argmax(probs)) == SUPPORTED_IDX, "the model 'knows': supported token is the mode"

print("next-token distribution at T=1.0:")
for word, p in zip(ANSWER_VOCAB, probs.tolist()):
    tag = "supported" if word == ANSWER_VOCAB[SUPPORTED_IDX] else "wrong"
    print(f"   {word:>8} ({tag:>9}): p = {p:.4f}")
print(f"\ntotal mass on WRONG tokens = {1 - probs[SUPPORTED_IDX].item():.4f}  (the hallucination base rate)")

next-token distribution at T=1.0:
     Austen (supported): p = 0.6343
     Bronte (    wrong): p = 0.1049
    Dickens (    wrong): p = 0.0858
      Eliot (    wrong): p = 0.0703
      Hardy (    wrong): p = 0.0575
    Gaskell (    wrong): p = 0.0471

total mass on WRONG tokens = 0.3657  (the hallucination base rate)


## Demo (a): hotter decoding hallucinates more

Temperature rescales logits before the softmax: $p_i(T) = \text{softmax}(z_i/T)_i$. Higher $T$ flattens
the distribution, bleeding mass off the supported token onto the wrong distractors. We measure the
**unsupported-claim rate** $R(T) = \sum_{i \in \text{wrong}} p_i(T)$ by Monte-Carlo sampling — and
assert it **rises** with $T$ before printing it.

In [3]:
rates = hallucination_rate_curve(base_logits_tensor(device=DEVICE), TEMPS_HALLUCINATION, device=DEVICE)

# CONTRACT: the unsupported-claim rate is monotone (non-decreasing) in temperature, and high T is
# far worse than low T. Assert BEFORE printing — a passing run is the proof.
for a, b in zip(rates, rates[1:]):
    assert b >= a - 0.01, "unsupported-claim rate must rise (not fall) with temperature"
assert rates[-1] > rates[0] + 0.2, "high temperature must hallucinate substantially more than low"

print("temperature -> unsupported-claim rate (ungrounded toy 'knowledge' sampler):")
for T, r in zip(TEMPS_HALLUCINATION, rates):
    bar = "#" * int(r * 50)
    print(f"   T={T:>3}:  {r:.3f}  {bar}")
print(f"\n=> rate climbs {rates[0]:.3f} (T={TEMPS_HALLUCINATION[0]}) -> "
      f"{rates[-1]:.3f} (T={TEMPS_HALLUCINATION[-1]}): hotter decoding hallucinates more.")

temperature -> unsupported-claim rate (ungrounded toy 'knowledge' sampler):
   T=0.3:  0.005  
   T=0.5:  0.067  ###
   T=0.7:  0.186  #########
   T=1.0:  0.364  ##################
   T=1.5:  0.540  ###########################
   T=2.0:  0.627  ###############################

=> rate climbs 0.005 (T=0.3) -> 0.627 (T=2.0): hotter decoding hallucinates more.


## Demo (b): grounding (retrieval) converts factuality into faithfulness

The strongest single lever against **factuality** hallucination is to stop asking the model to *recall*
and start asking it to *read*. Retrieval-augmented generation (RAG) puts a relevant passage in the
context. We model this at the logit level: grounding adds a boost to the **supported** token's logit.
The supported mass jumps, and the unsupported-claim rate drops **at every temperature**.

In [4]:
ungrounded = base_logits_tensor(device=DEVICE)
grounded = grounded_logits_tensor(device=DEVICE)   # +GROUNDING_BOOST on the supported token

p_ung = softmax_with_temperature(ungrounded, 1.0)[SUPPORTED_IDX].item()
p_grd = softmax_with_temperature(grounded, 1.0)[SUPPORTED_IDX].item()
print(f"supported-answer mass at T=1.0:  ungrounded {p_ung:.3f}  ->  grounded {p_grd:.3f}  "
      f"(+{GROUNDING_BOOST:.0f} logit)")

rates_u = hallucination_rate_curve(ungrounded, TEMPS_HALLUCINATION, device=DEVICE)
rates_g = hallucination_rate_curve(grounded, TEMPS_HALLUCINATION, device=DEVICE)

# CONTRACT: grounding never increases the rate, and substantially cuts it at T=1.0.
for u, g in zip(rates_u, rates_g):
    assert g <= u, "grounding must not increase the unsupported-claim rate"
i = TEMPS_HALLUCINATION.index(1.0)
assert rates_g[i] < rates_u[i] - 0.2, "grounding must substantially cut hallucination at T=1.0"

print("\ntemperature | ungrounded -> grounded:")
for T, u, g in zip(TEMPS_HALLUCINATION, rates_u, rates_g):
    print(f"   T={T:>3}:  {u:.3f}  ->  {g:.3f}")
print(f"\n=> at T=1.0 grounding cuts unsupported claims {rates_u[i]:.3f} -> {rates_g[i]:.3f} "
      f"(~{rates_u[i]/max(rates_g[i],1e-9):.0f}x fewer).")

supported-answer mass at T=1.0:  ungrounded 0.634  ->  grounded 0.990  (+4 logit)

temperature | ungrounded -> grounded:
   T=0.3:  0.005  ->  0.000
   T=0.5:  0.067  ->  0.000
   T=0.7:  0.186  ->  0.001
   T=1.0:  0.364  ->  0.011
   T=1.5:  0.540  ->  0.078
   T=2.0:  0.627  ->  0.188

=> at T=1.0 grounding cuts unsupported claims 0.364 -> 0.011 (~32x fewer).


**Why this is the taxonomy in action.** Ungrounded, "who wrote it?" is a *factuality* question (vs the
world). Grounded, it becomes "is this in the passage?" — a *faithfulness* question (vs the source),
which the model answers far better and which you can **verify** (cite the span). Grounding doesn't
delete error — it converts a hard-to-check factuality error into an easy-to-check faithfulness one.

## Demo (c): calibration & abstention — teach the model to say "I don't know"

A model is **calibrated** if its confidence equals its accuracy. Our toy "model" is deliberately
*mis*-calibrated (over-confident). We measure **Expected Calibration Error**
$\text{ECE} = \sum_b \frac{n_b}{N}\,|\text{acc}(b) - \text{conf}(b)|$, then **abstain** below a confidence
threshold $\tau$: answer only when confidence $\ge \tau$. As $\tau$ rises, answered-accuracy rises while
coverage falls — the risk–coverage trade.

In [5]:
confidence, correct = make_calibration_data()
ece = expected_calibration_error(confidence, correct)
base_acc = float(correct.mean())  # accuracy if we answer EVERYTHING

taus = np.array([0.5, 0.6, 0.7, 0.8, 0.9])
cov, acc = coverage_accuracy_curve(confidence, correct, taus)

# CONTRACT: raising tau lowers coverage and raises answered-accuracy; abstention beats answer-all.
for c0, c1 in zip(cov, cov[1:]):
    assert c1 <= c0 + 1e-9, "coverage must fall as tau rises"
valid = ~np.isnan(acc)
acc_valid = acc[valid]
for a0, a1 in zip(acc_valid, acc_valid[1:]):
    assert a1 >= a0 - 0.02, "answered-accuracy must rise as tau rises"
assert acc[-1] > base_acc, "the high-confidence answered set must beat answer-everything accuracy"

print(f"answer-everything accuracy = {base_acc:.3f}   ECE = {ece:.3f}  (large ECE = confidently wrong)")
print("\n tau | coverage | answered-accuracy")
for t, c, a in zip(taus, cov, acc):
    print(f" {t:.1f} |  {c:.3f}   |   {a:.3f}")
print(f"\n=> abstaining at tau=0.9 lifts accuracy {base_acc:.3f} -> {acc[-1]:.3f} "
      f"while coverage falls 1.000 -> {cov[-1]:.3f}.")

answer-everything accuracy = 0.506   ECE = 0.242  (large ECE = confidently wrong)

 tau | coverage | answered-accuracy
 0.5 |  1.000   |   0.506
 0.6 |  0.792   |   0.599
 0.7 |  0.596   |   0.687
 0.8 |  0.392   |   0.775
 0.9 |  0.199   |   0.858

=> abstaining at tau=0.9 lifts accuracy 0.506 -> 0.858 while coverage falls 1.000 -> 0.199.


**The catch:** abstention only helps if confidence is *correlated* with correctness. On an
*anti*-calibrated model (confident exactly when wrong), thresholding on confidence makes things
**worse** — you'd keep the confident hallucinations. Always measure ECE / a reliability diagram before
shipping an abstention threshold. The threshold is only as good as the confidence it gates on.

## Demo (d): helpful vs harmless — the alignment Pareto frontier

The easiest way to be **harmless** is to **refuse more** — which makes the model **less helpful**
(over-refusal). A single permissiveness knob $r$ (answer when perceived harm $< r$) cannot maximize
both. Sweeping $r$ traces a **Pareto frontier**: helpfulness rises and harmlessness falls, and no single
$r$ aces both.

In [6]:
perceived_harm, is_harmful = make_request_population()
helpful, harmless = helpful_harmless_curve(perceived_harm, is_harmful, PERMISSIVENESS_THRESHOLDS)

# CONTRACT: helpfulness rises and harmlessness falls with r; no single knob aces BOTH.
assert helpful[-1] > helpful[0] + 0.5, "high permissiveness must be far more helpful than low"
assert harmless[0] > harmless[-1] + 0.5, "low permissiveness must be far more harmless than high"
both_high = (helpful >= 0.9) & (harmless >= 0.9)
assert not both_high.any(), "no single scalar threshold should ace both helpful AND harmless"

print("helpful vs harmless across the permissiveness knob r:")
print("    r  | helpfulness | harmlessness")
for r in (0.0, 0.25, 0.5, 0.75, 1.0):
    i = int(np.argmin(np.abs(PERMISSIVENESS_THRESHOLDS - r)))
    print(f"  {PERMISSIVENESS_THRESHOLDS[i]:.2f} |    {helpful[i]:.3f}    |    {harmless[i]:.3f}")
best_both = (np.minimum(helpful, harmless)).max()
print(f"\n=> best simultaneous min(helpful, harmless) over ALL r = {best_both:.3f} < 1.0: "
      f"one scalar knob cannot reach the top-right corner.")

helpful vs harmless across the permissiveness knob r:
    r  | helpfulness | harmlessness
  0.00 |    0.000    |    1.000
  0.25 |    0.356    |    1.000
  0.50 |    0.700    |    0.979
  0.75 |    0.947    |    0.712
  1.00 |    0.999    |    0.189

=> best simultaneous min(helpful, harmless) over ALL r = 0.871 < 1.0: one scalar knob cannot reach the top-right corner.


## Recap

- **(a)** The unsupported-claim rate is **monotone in temperature** — the softmax floor, made empirical.
- **(b)** Grounding (a retrieval boost) cuts hallucination **~33× at T=1.0** by moving mass onto the
  supported token — factuality converted into checkable faithfulness.
- **(c)** Abstaining at $\tau=0.9$ buys **+35 points of accuracy** (0.51 → 0.86) at the cost of coverage —
  the risk–coverage trade, valid only on a (weakly) calibrated model.
- **(d)** Helpfulness and harmlessness move in **opposite directions** under one knob — alignment is a
  genuine multi-objective **Pareto trade**, which over-tuning turns into **over-refusal**.

These are toy distributions, so trust the **shapes and directions** — every one matches a real, named
phenomenon. The capstone synthesis is on the [concept page](../20-Hallucination-and-Alignment-Basics.md):
hallucination is a property of the *whole* pipeline — born in the objective, shaped by the data,
amplified by decoding, taught or untaught in post-training, revealed only by honest evaluation — and the
four mitigations (**ground, decode, abstain, verify**) each attack a different part of it.